<a href="https://colab.research.google.com/github/imdann06/estructra_de_bases_de_datos/blob/main/Arboles/ejercicio_arboles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Tienes un archivo con 10,000 estudiantes (nombre, edad,
promedio). Necesitas implementar un sistema que
permita:
1. Buscar un estudiante por su ID (número de matrícula)
2. Insertar nuevos estudiantes
3. Listar todos los estudiantes en orden por ID


In [1]:
pip install Faker #Instalacion libreria Faker, esta se usa para generar nombres aleatorios con la funcion .first_name

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 24.9 MB/s eta 0:00:00


In [5]:
#app funcional

import random
from faker import Faker

def generar_estudiantes():
    fake = Faker()
    estudiantes = []

    for i in range(1, 10001):
        estudiante = {
            "id": i,
            "nombre": fake.first_name(),
            "promedio": round(random.uniform(0, 10), 1)
        }
        estudiantes.append(estudiante)

    return estudiantes


def buscar_estudiante(estudiantes):
    id_buscada = int(input("Ingrese el ID del estudiante a buscar: "))

    for estudiante in estudiantes:
        if estudiante["id"] == id_buscada:
            print(f'ID: {estudiante["id"]} | Nombre: {estudiante["nombre"]} | Promedio: {estudiante["promedio"]}')
            return

    print("Estudiante no encontrado")


def insertar_estudiante(estudiantes):
    id_est = len(estudiantes) + 1
    nombre = input("Ingrese nombre: ")
    promedio = float(input("Ingrese promedio: "))

    estudiantes.append({
        "id": id_est,
        "nombre": nombre,
        "promedio": promedio
    })


def listar_estudiantes(lista):
    for estudiante in lista:
        print(estudiante)


# generar los 10000 estudiantes
estudiantes = generar_estudiantes()


while True:

    print("\n--- MENU ---")
    print("1. Buscar estudiante")
    print("2. Insertar estudiante")
    print("3. Listar estudiantes")
    print("4. Salir")

    opcion = input("Seleccione una opción: ")

    if opcion == "1":
        buscar_estudiante(estudiantes)

    elif opcion == "2":
        insertar_estudiante(estudiantes)

    elif opcion == "3":
        listar_estudiantes(estudiantes)

    elif opcion == "4":
        print("Ha salido.")
        break

    else:
        print("Opción inválida")


--- MENU ---
1. Buscar estudiante
2. Insertar estudiante
3. Listar estudiantes
4. Salir
Seleccione una opción: 4
Ha salido.


In [13]:
#Tiempo de lista

import time

def medir_busquedas(estudiantes):

    ids_aleatorios = random.sample(range(1, len(estudiantes)+1), 1000)

    inicio = time.time()

    for id_buscado in ids_aleatorios:

        for estudiante in estudiantes:
            if estudiante["id"] == id_buscado:
                break

    fin = time.time()

    print("Tiempo para 100 búsquedas:", fin - inicio, "segundos")


In [14]:
#100 estudiantes Lista
medir_busquedas(estudiantes)

Tiempo para 100 búsquedas: 0.7373635768890381 segundos


In [15]:
#construccion arbol ABB
class Nodo:
    def __init__(self, estudiante):
        self.estudiante = estudiante
        self.izquierda = None
        self.derecha = None


def insertar(nodo, estudiante):
    if nodo is None:
        return Nodo(estudiante)

    actual = nodo
    while True:
        if estudiante["id"] < actual.estudiante["id"]:
            if actual.izquierda is None:
                actual.izquierda = Nodo(estudiante)
                return nodo
            actual = actual.izquierda
        else:
            if actual.derecha is None:
                actual.derecha = Nodo(estudiante)
                return nodo
            actual = actual.derecha


def buscar_abb(nodo, id_buscado):
    actual = nodo
    while actual is not None:
        if id_buscado == actual.estudiante["id"]:
            return actual.estudiante
        elif id_buscado < actual.estudiante["id"]:
            actual = actual.izquierda
        else:
            actual = actual.derecha
    return None


# construir el ABB
raiz = None

for estudiante in estudiantes:
    raiz = insertar(raiz, estudiante)


import time
import random

def medir_busquedas_abb(raiz):

    ids = random.sample(range(1,10001),1000)

    inicio = time.time()

    for i in ids:
        buscar_abb(raiz, i)

    fin = time.time()

    print("Tiempo ABB:", fin - inicio, "segundos")

In [16]:
#100 estudiantes ABB
medir_busquedas_abb(raiz)

Tiempo ABB: 0.005715608596801758 segundos


In [10]:
pip install bplustree # Instalación de la librería B-Plus Tree para Python.

  Preparing metadata (setup.py) ... done
  Created wheel for rwlock: filename=rwlock-0.0.7-py3-none-any.whl size=3309 sha256=67214ffe57b93e81ecd2c23290465d95a39e0c64681cdf35335cb54dd74a7e7c
  Stored in directory: /root/.cache/pip/wheels/55/ed/aa/dc27251f7300e63daa0fe70cb03b209480a60a70c31aae42bd
Successfully built rwlock


In [ ]:
"""
import os

if os.path.exists("/tmp/estudiantes.db"):
    os.remove("/tmp/estudiantes.db")

arbol_bplus = BPlusTree("/tmp/estudiantes.db", order=50)

import random
import time

def medir_busquedas_bplus(arbol):

    ids = random.sample(range(1,10001),100)

    inicio = time.time()

    for i in ids:
        arbol.get(i)

    fin = time.time()

    print("Tiempo B+:", fin - inicio, "segundos")

"""
#Solo funciona con 100 datos

In [17]:
import os
import random
import time
from bplustree import BPlusTree

# Definir la ruta en el entorno de Colab
ruta_db = "/tmp/estudiantes.db"

# 1. Limpieza total de archivos previos para evitar el AssertionError
for extension in ["", "-wal", "-shm"]:
    if os.path.exists(ruta_db + extension):
        os.remove(ruta_db + extension)

# 2. Inicializar el árbol
# El 'order' define cuántas llaves tiene cada nodo
arbol_bplus = BPlusTree(ruta_db, order=50)

# 3. ¡CRUCIAL! Insertar los datos antes de medir
# El error suele dar porque intentas buscar 1000 elementos en un árbol vacío
print("Insertando datos en el Árbol B+...")
for est in estudiantes:
    # La llave es el ID (int), el valor debe ser bytes
    arbol_bplus.insert(est["id"], est["nombre"].encode('utf-8'))

def medir_busquedas_bplus(arbol, cantidad=1000):
    # Seleccionamos IDs que sabemos que existen
    ids = random.sample(range(1, 10001), cantidad)

    inicio = time.time()
    for i in ids:
        try:
            arbol.get(i)
        except KeyError:
            continue
    fin = time.time()

    print(f"Tiempo B+ ({cantidad} búsquedas): {fin - inicio:.6f} segundos")

# 4. Ejecutar la medición
medir_busquedas_bplus(arbol_bplus, 1000)

# 5. SIEMPRE cerrar el árbol al terminar para liberar los archivos
arbol_bplus.close()

Insertando datos en el Árbol B+...
Tiempo B+ (1000 búsquedas): 0.140640 segundos


In [12]:
random.shuffle(estudiantes) # desordena la lista de estudiantes

#COMPARACION DE TIEMPOS IDS ORDENADAS VS IDS EN DESORDEN PARA 1000 BUSQUEDAS


ID ORDENADA

Lista: Tiempo para 1000 búsquedas: 0.028342723846435547 segundos

ABB: Tiempo ABB:0.24692916870117188 segundos

B+: : Tiempo B+: 0.002257108688354492 segundos

ID DESORDENADA:

Lista: Tiempo para 1000 búsquedas: 0.7373635768890381 segundos

ABB: Tiempo ABB: 0.0008308887481689453 segundos

B+: Tiempo B+: 0.0020246505737304688 segundos

#COMPARACION DE TIEMPOS IDS ORDENADAS VS IDS EN DESORDEN PARA 1000 BUSQUEDAS

ID ORDENADA

Lista: Tiempo para 100 búsquedas: 0.20276355743408203 segundos
ABB: Tiempo ABB:0.24692916870117188 segundos

B+: : Tiempo B+: 0.109117 segundos
ID DESORDENADA:

Lista: Tiempo para 100 búsquedas: 0.10222601890563965 segundos

ABB: Tiempo ABB: 0.005715608596801758 segundos

B+: Tiempo B+ (1000 búsquedas): 0.140640 segundos